<a href="https://colab.research.google.com/github/Pitou11/fairness2/blob/main/UPP26/TP_Privacy/TP_privacy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TP Privacy

L'objectif de ce TP est d'implémenter et experimenter des mécanismes utilisés en Privacy.

## 1. Implémentation des méchanismes vérifiant la DP

In [13]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn as sk

### 1.1. Randomized Response
#### 1.1.1. Implémenter une fonction prenant en entrée une proportion p et un nombre d'observation à générer nb, et qui renvoie une liste de nb réalisation de Bernouilli de paramètre p.

In [4]:
def Ber(n, p):
  return np.random.binomial(n, p, 1)

n,p = 100, 0.25
Ber(n,p)

array([21])

#### 1.1.2. Utiliser la fonction implémentée pour générer 100 observations avec une proportion égale à 0.25

#### 1.1.3. Implémenter une fonction permettant d'appliquer le mécansime de réponse randomisée.

Pour rappel, celui-ci est défini de la manière suivante:

Toss a coin, if heads, then toss the coin again (ignoring the outcome), and answer
the question honestly, if tails, then toss the coin again and answer “Yes” if heads,
“No” if tails

In [7]:
def rep_rand(q):
  first_coin = np.random.choice([0, 1])

  if first_coin == 0:
    return q
  else:
    second_coin = np.random.choice([0, 1])
    return int(second_coin)

rep_rand(1)

0

#### 1.1.4. Montrer que le mécansime respecte la (ln 3, 0) -DP

Indice: Pour un répondant donné, quelles sont les probabilités que la réponse soit 1 alors que la vraie information est 1 et que la réponse soit 0 alors que la vraie information est 1 ?

In [11]:
p=0
p2=0
for i in range(100):
  p += rep_rand(1)
p = p/100
print("proba que la rep soit 1 alors que vrai info c'est 1")
print(p)
for i in range(100):
  p2 += rep_rand(1)
p2 = (100 - p2)/100
print("proba que la rep soit 0 alors que vrai info c'est 1")
print(p2)

print("la proba d'avoir 1 sachant que la rep est 0,75 et sinon c'est 0,25")

proba que la rep soit 1 alors que vrai info c'est 1
0.71
proba que la rep soit 0 alors que vrai info c'est 1
0.25
1.0438040521731147


#### 1.1.5. Appliqur le mécanisme implémenté sur la liste d'observations générées en 1.1.2. A partir de version randomisée des observations, estimé la proportion de 1.


### 1.2. Mécanisme de Laplace

#### 1.2.1. Charger le jeu de données diabetes (présent notamment dans sklearn). Etudier rapidement le jeu de données (premières observation, stat descriptives simples, etc.)



In [24]:
data = sk.datasets.load_diabetes()
df = pd.DataFrame(data=data.data, columns=data.feature_names)
df.head()
#dataset déjà normalisé

,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022688,-0.009362
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641


In [20]:
data['age'].describe()

,age
count,4.420000e+02
mean,-2.511817e-19
std,4.761905e-02
min,-1.072256e-01
25%,-3.729927e-02
50%,5.383060e-03
75%,3.807591e-02
max,1.107267e-01


In [43]:
if not(np.any(np.isnan(df))):
  print("aucune valeur manquante ou nan")
else:
  print("il y a des valeurs manquantes")

ages = np.unique(df['age'])
print(f"il y a {len(ages)} ages différents")

sex = np.unique(df['sex'])
print(f"il y a {len(sex)} sexes différents :\n      {sex} ")

#bmi = np.unique(df['bmi'])
#print(f"il y a {len(bmi)} bmi différents")

aucune valeur manquante ou nan
il y a 58 ages différents
il y a 2 sexes différents :
      [-0.04464164  0.05068012] 


on peux supposer qu'il n'y a que le genre Feminin et Masculin -> 2 sexes différents, et pas de personnes qui n'ont pas donner leur genre

#### 1.2.2. Implémenter des fonctions pour faire des requêtes sans contraintes de Privacy

+ Requête pour compter le nombre d'individus avec une caractéristique donnée: prend en entrée une dataframe, un atttribut catégorique (e.g. "sex") et une valeur (e.g. "2"), et renvoie le nombre d'individus ayant comme genre 2.
+ Requête pour moyenner une variable numérique: prend en entrée un jeu de données et une variable numérique et retourne la valeur moyenne de celles-ci

In [47]:
np.count_nonzero(df['sex'])

442

#### 1.2.3. Implémenter une fonction permettant d'appliquer un mécanisme de Laplace permettant de compter le nombre d'individus avec une caractéristique donnée. Le budget de privacy, la dataframe et le nom de la caractéritique sont les paramètres de la fonction.

Appliquer le mécanisme sur la variable "sex". Quelle sera la sensibilité ?

Estimer la norme 1 empirique entre le count avec privacy et sans privacy. Tracer cette valeur en fonction de la valeur du budget de privacy. Que constatez-vous?



#### 1.2.4. Implémenter une fonction permettant d'appliquer un mécanisme de Laplace permettant de moyenner une variable numérique. Le budget de privacy, la dataframe et le nom de la caractéritiques sont les paramètres de la fonction.

Appliquer le mécanisme sur la variable BMI. Quelle est la sensibilité?

Estimer la norme 1 empirique entre le count avec privacy et sans privacy. Tracer cette valeur en fonction de la valeur du budget de privacy. Que constatez-vous?

#### 1.2.5. Montrer que le mécanisme de Laplace respecte la epsilon DP

### 1.3. Mécanisme Gaussien

#### 1.3.1 Implémenter le mécanisme Gaussien.

#### 1.3.2. Comparer le compromis utilité/niveau de Privacy avec le mécanisme de Laplace

## 2. Local Differential Privacy

### 2.1. K-ary Randomized Response

#### 2.1.1. Implémenter une fonction prenant en entrée une liste de classes et les effectifs de chaque classe et génére un vecteur où chaque classe est répétée pour respecter son effectif. Par exemple, si la fonction prend en entrée (class_1, class_2, class_3) et (1, 3, 2), la sortie attednue est [class_1, class_2, class_2, class_2, class_3, class_3 ]

#### 2.1.2. Implémenter une fonction permettant le mécanisme K-ary randomized response prenant en entrée l'information a randomizée et le budget de Privacy

Pour rappel, le mécanisme est donné par ![kary](https://github.com/alheliou/Bias_mitigation/blob/main/UPP26/TP_Privacy/kary.png?raw=1)

#### 2.1.3. Implémenter une fonction permettant d'estimer sans biais l'histogramme à partir de la version des données randomizées

Pour rappel: ![kary2](https://github.com/alheliou/Bias_mitigation/blob/main/UPP26/TP_Privacy/kary2.png?raw=1)

#### 2.1.4 Considérer un vecteur à 10 classes avec respectivement 19%, 13%, 12%, 11%, 10%, 9%, 8%, 7%, 6%, et 5%. Faire varier la taille totale des effectifs entre 1000 et 200000. Pour un budget de Privacy de 1, calculer l'histogramme sur les données non randomizées, l'histogramme calculer naivement sur les données randomizées et l'histogramme obtenu avec l'estimateur sans biais. Que constatez-vous ?

#### 2.1.5. Tracer pour toutes les classes l'estimateur de l'histogramme calculé naivement sur les données randomizées moins la vraie valeur de l'histogramme au carré en fonction du nombre d'observations. Que constatez vous ?

#### 2.1.6. Tracer pour toutes les classes l'estimateur de l'histogramme calculé avec l'estimateur sans biais sur les données randomizées moins la vraie valeur de l'histogramme au carréen fonction du nombre d'observations. Que constatez vous ?

### 2.2. En se replaçant dans le contexte centralisé, appliquer les mécanismes de Laplace et Gaussien pour estimer les histogrammes. Comment évoluent les erreurs en fonction du nombre d'observations ? Quel mécanisme protège plus la privacy entre le mécanisme étudié dans la question 2.1 et les mécanismes de Laplace et Gaussien?